In [ ]:
from matplotlib import pyplot as plt
from scipy.stats import norm
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import pandas as pd
import seaborn as sns

sns.set_theme(style="darkgrid")
xdata = []
ydata = []

In [ ]:
with open('/users/cdcook/VSP/datafiles/PSdata/xtetrans_1b_Exp4Final.csv', newline='') as w:
    data = list(csv.reader(w))
    data.pop(0)

In [ ]:
def colorSub(data, kronCut, kronDist, kronBand, bitFlagsTF, bitFlags, band1a, band1b, band2a, band2b, max):
    xdata = []
    ydata = []
    counter = 0
    if (bitFlagsTF):
        if (kronCut):
            for n in range(8090):
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0 and np.binary_repr(int(data[n][5]), width=8) == bitFlags and float(data[n][kronBand]) - float(data[n][kronBand + 1]) < kronDist and float(data[n][kronBand]) - float(data[n][kronBand + 1]) > -1*kronDist:
                    counter = counter + 1
                    x = float(data[n][band1a]) - float(data[n][band1b])
                    xdata.append(x)
                        
                    y = float(data[n][band2a]) - float(data[n][band2b])
                    ydata.append(y)
                                   
        else:
            for n in range(8090):
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0 and np.binary_repr(int(data[n][5]), width=8) == bitFlags:
                    counter = counter + 1
                    x = float(data[n][band1a]) - float(data[n][band1b])
                    xdata.append(x)
                        
                    y = float(data[n][band2a]) - float(data[n][band2b])
                    ydata.append(y)
    else:
        if (kronCut):
            for n in range(8090):
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0 and float(data[n][kronBand]) - float(data[n][kronBand + 1]) < kronDist and float(data[n][kronBand]) - float(data[n][kronBand + 1]) > -1*kronDist:
                    counter = counter + 1
                    x = float(data[n][band1a]) - float(data[n][band1b])
                    xdata.append(x)
                        
                    y = float(data[n][band2a]) - float(data[n][band2b])
                    ydata.append(y)
                                   
        else:
            for n in range(8090):
                if float(data[n][4]) > 5.0 and float(data[n][4]) < 25.0:
                    counter = counter + 1
                    x = float(data[n][band1a]) - float(data[n][band1b])
                    xdata.append(x)
                        
                    y = float(data[n][band2a]) - float(data[n][band2b])
                    ydata.append(y)
    return xdata, ydata, counter

In [ ]:
kronCut = False
kronDist = 0.5
kronBand = 6
bitFlagsTF = False
bitFlags = '00000000'
sub1 = 'ri'
sub2 = 'gr'

if (sub1 == 'gr'):
    band1a = 6
    band1b = 8
    band1name = "g-r"
    
elif (sub1 == 'gi'):
    band1a = 6
    band1b = 10
    band1name = "g-i"
    
elif (sub1 == 'ri'):
    band1a = 8
    band1b = 10
    band1name = "r-i"
    
if (sub2 == 'gr'):
    band2a = 6
    band2b = 8
    band2name = "g-r"

elif (sub2 == 'gi'):
    band2a = 6
    band2b = 10
    band2name = "g-i"
    
elif (sub2 == 'ri'):
    band2a = 8
    band2b = 10
    band2name = "r-i"

if (kronBand == 6):
    kronName = 'g'
if (kronBand == 8):
    kronName = 'r'
if (kronBand == 10):
    kronName = 'i'
if (kronBand == 12):
    kronName = 'z'
if (kronBand == 14):
    kronName = 'y'

In [ ]:
xdata, ydata, counter = colorSub(data, kronCut, kronDist, kronBand, bitFlagsTF, bitFlags, band1a, band1b, band2a, band2b, 8090)

In [ ]:
params = np.polyfit(xdata, ydata, 1, full=False, cov=True)
difList = []    
n = 0
for i in ydata:
    difList.append(i - (vsp.oneDFit(params[0][0], params[0][1], xdata[n])))
    n = n + 1

dataDf = pd.DataFrame({'x': xdata, 'y': ydata})
print(dataDf.head())

print(params[0][0], params[0][1])
print("Total Count: ", counter)

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(x="x", y="y", data=dataDf)
plt.xlabel(band1name, fontsize=20)
plt.ylabel(band2name, fontsize=20)
plt.xlim(-1,3)
plt.ylim(-1,3)
plt.title("Color plot of " + band2name + " over " + band1name + " " + kronName + "kron cut: +-" + str(kronDist) + " -- no e-flags", fontsize=20)
plt.show()

In [ ]:
dataDf = dataDf.drop(dataDf[dataDf['x'] > 1.5].index)
dataDf = dataDf.drop(dataDf[dataDf['x'] < -0.2].index)

plt.figure(figsize=(18, 9))
sns.scatterplot(x="x", y="y", data=dataDf)
plt.xlim(-1,3)
plt.ylim(-1,3)
plt.xlabel(band1name, fontsize=20)
plt.ylabel(band2name, fontsize=20)
plt.title("After bound cut", fontsize=20)
plt.show()

In [ ]:
xframe = dataDf['x']
yframe = dataDf['y']
#print(xframe, yframe)

fit = np.poly1d(np.polyfit(xframe, yframe, 10))
print(fit)

#print("Total count before: ", dataDf['x'].count())
dataDf = dataDf.drop(dataDf[abs(dataDf['y'] - fit(dataDf['x'])) > 0.2].index)

#print("Total count after: ", dataDf['x'].count())

In [ ]:
plt.figure(figsize=(18, 9))
sns.scatterplot(x="x", y="y", data=dataDf)
plt.xlim(-1,3)
plt.ylim(-1,3)
plt.plot(np.unique(xdata), fit(np.unique(xdata)), color='red')
#plt.plot(np.unique(xdata), fit(np.unique(xdata)) + 0.2, color='green')
#plt.plot(np.unique(xdata), fit(np.unique(xdata)) - 0.2, color='green')
plt.xlabel(band1name, fontsize=20)
plt.ylabel(band2name, fontsize=20)
plt.title("After polyfit +- 0.2 cut", fontsize=20)
plt.show()